## Task 5 (legacy): SPN mount

In [0]:
login = "lena066636"
scope_name = f"{login}-scope"

client_id = dbutils.secrets.get(scope=scope_name, key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-databricks-adls-appkey")
tenant_id = dbutils.secrets.get(scope=scope_name, key="tenant-id")


In [0]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

try:
    dbutils.fs.mount(
        source=f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/",
        mount_point=f"/mnt/{login}",
        extra_configs=configs)
except Exception as e:
    print(e)


In [0]:
spark.conf.set("fs.azure.account.auth.type", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id", client_id)
spark.conf.set("fs.azure.account.oauth2.client.secret", client_secret)
spark.conf.set("fs.azure.account.oauth2.client.endpoint", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

dbutils.fs.ls(f"abfss://{login}@dlsua5816bd.dfs.core.windows.net/")


**Result:** both `dbutils.fs.mount()` and `spark.conf.set("fs.azure.account.auth.type", ...)` are blocked on this Shared UC cluster (`not whitelisted`, `CONFIG_NOT_AVAILABLE`). Direct SPN auth is not possible here at all - the UC External Location from Task 2-3 is the only working path on this cluster type.